In [4]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

In [5]:
pc = Pinecone(api_key=os.getenv("pinecone_api_key"))

In [5]:
pc.create_index(
  name="social-media",
  dimension=768,
  metric="cosine",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  )
)

{
    "name": "social-media",
    "metric": "cosine",
    "host": "social-media-ph2q0am.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "20

In [11]:
index_name = "social-media"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

# 3. Generate and Insert Embeddings
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
text_data = ["What is quantum computing?", "How does machine learning work?"]
embeddings = model.encode(text_data).tolist() # Convert to list for Pinecone

# # Upsert with IDs and optional metadata
# vectors = [
#     {"id": f"vec{i}", "values": emb, "metadata": {"text": text}} 
#     for i, (emb, text) in enumerate(zip(embeddings, text_data))
# ]
# index.upsert(vectors=vectors)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11097.81it/s]


In [12]:
vectors_to_upsert = [
    {
        "id": "item_1005",  # Your specific custom ID
        "values": embeddings[0], 
        "metadata": {
            "text": "what is quantum computing?"
        }
    },
    {
        "id": "item_1006",
        "values": embeddings[1],
        "metadata": {
            "text": "How does machine learning work?"
        }
    }
]

index.upsert(vectors=vectors_to_upsert)

UpsertResponse(upserted_count=2, _response_info={'raw_headers': {'date': 'Wed, 29 Apr 2026 04:48:17 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '6', 'x-pinecone-request-logical-size': '3171', 'x-pinecone-request-latency-ms': '395', 'x-envoy-upstream-service-time': '166', 'x-pinecone-response-duration-ms': '399', 'grpc-status': '0', 'server': 'envoy'}})

## Retrieve

In [25]:
query_text = "Learning machine learning"
query_vector = model.encode(query_text).tolist()

In [26]:
# 2. Query Pinecone for the top matches
results = index.query(
    vector=query_vector,
    top_k=3,                  # Number of related results to get
    include_values=True,      # Returns the raw embedding values
    include_metadata=True     # Returns your custom ID and stored text
)

In [27]:
for match in results['matches']:
    print(f"ID: {match['id']}")
    print(f"Score (Similarity): {match['score']}")
    print(f"Stored Text: {match['metadata']['text']}")

ID: item_1006
Score (Similarity): 0.820980072
Stored Text: How does machine learning work?
ID: 299909109209
Score (Similarity): 0.51619339
Stored Text: Testing tweet for fastapi only
ID: item_1004
Score (Similarity): 0.513048232
Stored Text: How come we are not getting any likes on our posts?
